In [14]:
import numpy as np
import h5py as h5
import geopandas as gpd
import pandas as pd
from itertools import product

In [15]:
time_span = 100
# max_resolution = 27000  # Maximum grid spacing of current sites
max_variance = 0.1  # m
probability = 1
split_method = 'cube'  # 'cube'
input_file = "..\\crustal\\discretised_CFM\\crustal_site_locations_national_9km_3km.geojson"
all_site_geojson = "..\\crustal\\discretised_CFM\\crustal_site_locations_national_1km.geojson"
PPE_file = "..\\results\\CFM\\sites_c_NjI5\\N46_b1089_C42_S10_TD_geodetic_c_NjI5_cumu_PPE.h5"
input_file = "..\\subduction\\discretised_puysegur\\py_site_locations_national_9kmS_3km.geojson"
all_site_geojson = "..\\subduction\\discretised_puysegur\\py_site_locations_national_1kmS.geojson"
PPE_file = "..\\results\\puysegur\\sites_py_M5NQ\\N46_b0902_C4_S10_ti_geodetic_py_M5NQ_cumu_PPE.h5"

In [16]:
split_factor = 3 if split_method == 'cube' else 2
sites = gpd.read_file(input_file)
higher_res_sites = gpd.read_file(all_site_geojson)
higher_res_sites = pd.concat([sites, higher_res_sites], ignore_index=True).drop_duplicates(subset='siteId', keep='first')
max_resolution = np.hstack([np.diff(np.unique(sites.geometry.x)), np.diff(np.unique(sites.geometry.y))]).max()
print(f"Max resolution of dataset: {max_resolution:.0f} m")

Max resolution of dataset: 9000 m


In [17]:
displacements = []
n_sites = sites.shape[0]
threshlims = np.array([0, 0, 1])
thresholds = np.arange(threshlims[0], threshlims[1], threshlims[2])
probability = round(probability * 1e-2, 4)

with h5.File(PPE_file, "r") as results_h5:
    for ix, site in enumerate(sites['siteId'], 1):
        print(f"{ix}/{n_sites}", end='\r')
        if not (threshlims == results_h5[site][str(time_span)]['thresh_para'][:]).all():
            threshlims = results_h5[site][str(time_span)]['thresh_para'][:]
            thresholds = np.arange(threshlims[0], threshlims[1], threshlims[2])
            print(f'\nNew thresholds at {site}: {threshlims}')
        exceedance_index = next((index for index, value in enumerate(results_h5[site][str(time_span)]['exceedance_probs_total_abs'][:]) if value <= probability), -1)
        displacements.append(thresholds[exceedance_index])
print('')
sites['disp'] = displacements

1/3339
New thresholds at 1090000_4883000: [0.e+00 3.e+01 1.e-02]
3339/3339


In [18]:
max_distance = np.sqrt(max_resolution ** 2 + max_resolution ** 2)
min_spacing = max_resolution
new_sites = set()
for ix, site in sites.iterrows():
    print(f"{ix}/{n_sites}", end='\r')
    new_id, split_site = [site['siteId']], False
    subset = sites[sites.geometry.distance(site.geometry) <= max_distance].drop([ix])
    if subset.empty:
        spacing = max_resolution
        split_site = True
    else:
        subset['weight'] = 1 / ((subset.geometry.distance(site.geometry) * 1e-3) ** 2)
        diffs = []
        for n in subset.index.tolist():
            diffs.append(abs(sites.iloc[n]['disp'] - site['disp']))
        subset['diff'] = diffs
        cumulative_weight = np.cumsum(subset['weight'].iloc[np.argsort(subset['weight'])] / subset['weight'].sum())
        weighted_median = subset.loc[cumulative_weight[cumulative_weight >= 0.5].index[0]]['diff']
        
        if weighted_median > max_variance:
            split_site = True
   
    if split_site:
        if not subset.empty:
            spacing = np.hstack([np.diff(np.unique(subset.geometry.x)), np.diff(np.unique(subset.geometry.y))]).min()
            min_spacing = min(min_spacing, spacing)
        x_range = np.arange(site.geometry.x -  2 * spacing / 3, site.geometry.x + 2 * spacing / 3 + 1, spacing / split_factor)
        y_range = np.arange(site.geometry.y - 2 * spacing / 3, site.geometry.y + 2 * spacing / 3 + 1, spacing / split_factor)
        for xx, yy in product(x_range, y_range):
            new_id.append(f"{xx:.0f}_{yy:.0f}")

    new_sites.update(new_id)

new_sites = list(new_sites)
outfile = input_file.replace('.geojson', f"_{min_spacing/(1000 * split_factor):.0f}km.geojson")
out_sites = higher_res_sites[higher_res_sites['siteId'].isin(new_sites)]
out_sites.to_file(outfile, driver="GeoJSON")
print(f'\n{out_sites.shape[0]} sites written to {outfile}')

3338/3339
13066 sites written to ..\subduction\discretised_puysegur\py_site_locations_national_9kmS_3km_1km.geojson
